# MTDC3T : reseau continu a trois terminaux, trois zones alternatives

La suite de `hvdc2t` : un troisieme terminal, et un reseau continu qui n'est
plus une simple liaison mais un **graphe**. Les trois zones alternatives sont
independantes ; elles n'echangent que par le continu.

```
   zone A                  bus DC 10 (type 2)              zone B
  IB1 -- 150 km -- bus 2 ------- GFM3DC, mode PV
                                 /            \
                    cable 100 km              cable 120 km
                        /                            \
        bus DC 11 (type 1, slack)              bus DC 12 (type 2)
             GFL2DC, mode VdcQ                  GFL2DC, mode PQ
                  |                                   |
        bus 3 -- 150 km -- IB4                bus 5 -- 150 km -- IB6
        (charge 40 MW)         zone B         (charge 50 MW)      zone C
```

**Les trois modes du lot y sont exerces a la fois** : `PV` (grid-forming a
puissance programmee, qui alimente le reseau continu), `VdcQ` (grid-following
qui tient la tension continue -- l'unique reference de l'ile) et `PQ`
(grid-following a puissance programmee, dont la tension continue sort du
bilan). Le Newton a donc trois inconnues continues de natures differentes :
deux tensions (noeuds 10 et 12) et une puissance alternative (celle du VSC du
bus 3).

**Le condensateur de chaque noeud** vaut la somme des demi-capacites des
cables qui y aboutissent, des `Cdc_pu` des convertisseurs raccordes et des
shunts -- le decalque exact de ce que `Cbus` fait deja du cote alternatif.
C'est `g2egrid.tables` qui le calcule, plus l'appelant.

Base systeme : **100 MVA, 50 Hz**. Base continue : **200 kV**.

In [ ]:
import sys
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Le paquet g2egrid est a la racine du projet (dossier parent de reseaux_test/).
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

from g2egrid.tables import Tables, assembler  # noqa: E402
from g2egrid.temporel import ModeleNonLineaire  # noqa: E402

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

## 1. Construction du reseau

### 1.1 Donnees

In [ ]:
NOM_RESEAU = "MTDC3T"
F_ZOOM = 5.0

f0 = 50.0
Sb = 100e6
UDC = 200e3

# BusType : 1 = slack, 2 = PV, 3 = PQ
Y_DER = [ # [BusID, Type, Un_V, V_pu, Theta_rad,   Pg_W, Qg_var,  Pl_W, Ql_var]
        (1, 1, 230e3, 1.00, 0.0,      0, 0,     0,     0),
        (2, 2, 230e3, 1.00, 0.0, -70e6, 0,     0,     0),
        (3, 3, 230e3, 1.00, 0.0,      0, 0,     0,     0),
        (4, 1, 230e3, 1.00, 0.0,      0, 0, 40e6, 12e6),
        (5, 3, 230e3, 1.00, 0.0,  25e6, 0,     0,     0),
        (6, 1, 230e3, 1.00, 0.0,      0, 0, 50e6, 15e6),
]

# T_Lines_SI : [ID, From, To, Length_km, R_ohm/km, X_ohm/km, B_S/km]
T_Lines_SI = [
    (1, 1, 2, 150.0, 0.03, 0.30, 4.0e-6),
    (2, 3, 4, 150.0, 0.03, 0.30, 4.0e-6),
    (3, 5, 6, 150.0, 0.03, 0.30, 4.0e-6),
]

# T_IBs : [ID, BusID, Sn_VA, E_pu, theta_deg, R_pu, X_pu]
T_IBs = [
    (1, 1, 1000e6, 1.00, 0, 0.005, 0.05),
    (2, 4, 1000e6, 1.00, 0, 0.005, 0.05),
    (3, 6, 1000e6, 1.00, 0, 0.005, 0.05),
]

# [ID, BusID, Sn_VA, Vdc_V]
T_GFM3s = [(1, 2, 200e6, UDC)]
T_GFL2s = [(1, 3, 200e6, UDC), (2, 5, 200e6, UDC)]

# ---------------------------------------------------------------- le continu
# T_BusDC : [BusID, Type, Un_V, V_pu, Pinj_W]
T_BusDC = [
    (10, 2, UDC, 1.00, 0.0),
    (11, 1, UDC, 1.00, 0.0),
    (12, 2, UDC, 1.00, 0.0),
]

# T_LinesDC : [ID, From, To, Length_km, R_ohm/km, L_mH/km, C_uF/km]
T_LinesDC = [
    (1, 10, 11, 100.0, 0.050, 0.35, 0.012),
    (2, 10, 12, 120.0, 0.050, 0.35, 0.012),
]

# T_ShuntsDC : [ID, BusID, C_pu]
T_ShuntsDC = [(1, 10, 0.5), (2, 11, 0.5), (3, 12, 0.5)]

# T_VSCs : [ID, BusAC, BusDC, Mode]
T_VSCs = [
    (1, 2, 10, "PV"),
    (2, 3, 11, "VdcQ"),
    (3, 5, 12, "PQ"),
]

# T_Lines_pu, T_Shunts, T_Transfos, T_GFM1s/2s/4s, T_GFL1s : vides ici.
T_SMs = []
T_Loads = []

# Reglages des convertisseurs. Les lignes alternatives font 150 km : plus
# courtes, le mode d'interaction du grid-forming avec sa ligne passe a droite
# de l'axe imaginaire (mesure : +5,6 s^-1 a 50 km).
GFM_PARAMS = dict(mp=0.02, nq=0.05, tau_VL=15e-3, tau_CL=0.5e-3)
GFL_PARAMS = dict(tau_CL=5e-3, tau_VL=0.02, tau_PLL=0.02)

### 1.2 Assemblage

In [ ]:
# L'assemblage passe par ``g2egrid.tables.assembler``, et non par une cellule
# recopiee : le reseau continu ajoute quatre tables, une capacite de noeud a
# calculer et une resolution de modes, et deux exemplaires de cette logique
# divergeraient au premier reglage. Les tables ci-dessus RESTENT la
# description du reseau ; seule la mecanique est appelee.
t = Tables(
    nom=NOM_RESEAU,
    f0=f0,
    Sb=Sb,
    Y_DER=Y_DER,
    T_Lines_SI=T_Lines_SI,
    T_IBs=T_IBs,
    T_SMs=T_SMs,
    T_GFM3s=T_GFM3s,
    T_GFL2s=T_GFL2s,
    T_Loads=T_Loads,
    GFM_PARAMS=GFM_PARAMS,
    GFL_PARAMS=GFL_PARAMS,
    T_BusDC=T_BusDC,
    T_LinesDC=T_LinesDC,
    T_ShuntsDC=T_ShuntsDC,
    T_VSCs=T_VSCs,
)
net = assembler(t)

print("Noeuds continus (capacite pu, base systeme) :")
for b, s in net.bus_specs_dc.items():
    genre = "slack continu" if s.btype == 1 else "P impose"
    print(f"  bus {b} : Un = {s.Un / 1e3:.0f} kV, C = {s.C:.4f} pu, {genre}")
print("\nModes des convertisseurs (par bus ALTERNATIF) :", net.vsc_modes)

## 2. Load flow AC/DC unifie

Un SEUL Newton sur `[theta_AC, V_AC, V_DC, P des VSC en mode Vdc...]`. Le couplage est `P_dc = -V_dc . i_port`, avec le `i_port` du modele dynamique : les pertes qui en sortent sont celles du filtre et du condensateur, jamais un rendement pose a la main.

In [ ]:
lf = net.run_loadflow()
print(
    f"Load flow AC/DC : converge en {lf.iterations} iterations "
    f"(mismatch {lf.mismatch:.2e})\n"
)

print(f"{'bus':>4} {'|V| [pu]':>10} {'angle [deg]':>12} "
      f"{'P inj [pu]':>12} {'Q inj [pu]':>12}")
print("-" * 54)
for b, i in net._bus_index().items():
    print(f"{b:>4} {abs(lf.V[i]):>10.4f} {np.degrees(np.angle(lf.V[i])):>12.4f} "
          f"{lf.S[i].real:>12.4f} {lf.S[i].imag:>12.4f}")

print(f"\n{'bus DC':>7} {'Vdc [pu]':>10} {'P des VSC [pu]':>16}")
print("-" * 36)
for j, b in enumerate(sorted(net.bus_specs_dc)):
    print(f"{b:>7} {lf.Vdc[j]:>10.6f} {lf.Pdc[j]:>16.6f}")
print(f"\nCorrection du P programme des VSC en mode Vdc... : {lf.P_vsc}")

### 2.1 Le bilan de puissance boucle

`somme(P_ac des VSC) + pertes des convertisseurs + pertes des cables = 0`. C'est l'epreuve qui dit que le point de fonctionnement mixte est coherent : le load flow et le modele dynamique comptent les MEMES pertes.

In [ ]:
# Le bilan de puissance de la liaison, PERTES CALCULEES A PART.
#
# Les pertes ne sont pas deduites du bilan (ce serait une tautologie) : elles
# sont recalculees depuis les ETATS du modele, ``R1.|is|^2 + R2.|ig|^2 +
# Gdc.vdc^2``. Si le load flow avait pose un rendement quelconque au lieu du
# vrai courant de port, c'est ici que cela se verrait.
net.solve_equilibrium()

prog = net._injections_programmees()
idx = net._bus_index()
tot_ac = tot_pertes = 0.0
print(f"{'convertisseur':<16} {'P_ac [pu]':>12} {'pertes [pu]':>14}")
print("-" * 44)
for el in net._elements_vsc():
    p = el.pdict
    x = dict(zip(el.states, el.x0))
    vdc = el.u0[el.port_dc_input_idx(0)]
    pertes = p["kS"] * (
        p["R1_pu"] * (x["isd"] ** 2 + x["isq"] ** 2)
        + p["R2_pu"] * (x["igd"] ** 2 + x["igq"] ** 2)
        + p["Gdc_pu"] * vdc**2
    )
    p_ac = lf.S[idx[el.buses[0]]].real - prog[el.buses[0]][0]
    tot_ac += p_ac
    tot_pertes += pertes
    print(f"{el.name:<16} {p_ac:>12.6f} {pertes:>14.6f}")

pos = {b: j for j, b in enumerate(sorted(net.bus_specs_dc))}
G = net._reseau_dc(pos)[0]
pertes_cables = float(lf.Vdc @ (G @ lf.Vdc))
print(f"\nSomme des P alternatifs  : {tot_ac:+.12f} pu")
print(f"Pertes des convertisseurs: {tot_pertes:+.12f} pu")
print(f"Pertes des cables continus: {pertes_cables:+.12f} pu")
print(f"BILAN (doit etre nul)    : {tot_ac + tot_pertes + pertes_cables:+.3e} pu")

## 3. Modele d'etat et analyse modale

`residu_equilibre` mesure si le point autour duquel on linearise est bien un point fixe du modele non lineaire complet -- la verification croisee que le lot 4 doit tenir.

In [ ]:
ss = net.build_state_space()
print(ss)
print(f"\n{len(ss.states)} etats, {len(ss.inputs)} entrees, {len(ss.outputs)} sorties")
print(f"stable : {ss.is_stable()}   max Re(lambda) = {ss.eig().real.max():+.4e}")

m = ModeleNonLineaire(net)
dx, dzu = m.residu_equilibre()
print(f"\nresidu_equilibre : |dx/dt| = {dx:.3e}, |z - z0| + |u - u0| = {dzu:.3e}")
print("  (le point de fonctionnement mixte est un point fixe du modele non lineaire)")

lam, Vr = np.linalg.eig(ss.A)
Wl = np.linalg.inv(Vr)
Pf = np.abs(Vr.T) * np.abs(Wl)
Pf = Pf / np.maximum(Pf.sum(axis=1, keepdims=True), 1e-30)
top3 = np.argsort(-Pf, axis=1)[:, :3]
k = np.arange(len(lam))
mag = np.abs(lam)
zeta = np.where(mag > 1e-10, -lam.real / np.where(mag > 1e-10, mag, 1.0), 0.0)

T_eig = pd.DataFrame({
    "Re": lam.real,
    "Im": lam.imag,
    "f_und [Hz]": mag / (2 * np.pi),
    "f_dam [Hz]": np.abs(lam.imag) / (2 * np.pi),
    "zeta [%]": 100 * zeta,
})
for j in range(3):
    T_eig[f"Etat {j + 1}"] = [ss.states[i] for i in top3[:, j]]
    T_eig[f"%{j + 1}"] = 100 * Pf[k, top3[:, j]]
T_eig = T_eig.iloc[np.argsort(-lam.real, kind="stable")]
T_eig.index.name = "Mode"
display(T_eig.head(20))

### 3.1 Carte des valeurs propres

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))

ax[0].scatter(lam.real, lam.imag / (2 * np.pi), s=22, marker="x")
ax[0].axvline(0.0, color="k", lw=0.8)
ax[0].set_xlabel("Re($\\lambda$) [1/s]")
ax[0].set_ylabel("Im($\\lambda$)/2$\\pi$ [Hz]")
ax[0].set_title(f"{NOM_RESEAU} - tous les modes ({len(lam)} etats)")
ax[0].grid(True, alpha=0.3)

zoom = np.abs(lam.imag) / (2 * np.pi) <= F_ZOOM
ax[1].scatter(lam[zoom].real, lam[zoom].imag / (2 * np.pi), s=32, marker="x")
for k2 in np.flatnonzero(zoom & (lam.imag > 1e-6)):
    ax[1].annotate(
        ss.states[top3[k2, 0]].split(".")[0],
        (lam[k2].real, lam[k2].imag / (2 * np.pi)),
        fontsize=7, xytext=(4, 3), textcoords="offset points",
    )
ax[1].axvline(0.0, color="k", lw=0.8)
ax[1].set_xlabel("Re($\\lambda$) [1/s]")
ax[1].set_ylabel("Im($\\lambda$)/2$\\pi$ [Hz]")
ax[1].set_title(f"modes lents (|f| < {F_ZOOM} Hz)")
ax[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()